# CS 3120/5120: Secure Distributed Computation
## Homework 5

In [ ]:
# Useful imports and utility functions
import pychor
import galois
import numpy as np
from dataclasses import dataclass
from typing import Dict

p = 2**31-1
GF = galois.GF(p)
GF_2 = galois.GF(2)

## Question 1 Setup

Below is the definition of `SecBitN`, a class for representing secret-shared bits using $n$-party additive secret sharing and performing operations using the $n$-party GMW protocol. In question 1, you'll implement the `protocol_gmw_mult_n` protocol.

In [ ]:
@pychor.local_function
def share_bit(secret, n):
    shares = [GF_2.Random() for _ in range(n-1)]
    shares.append(GF_2(secret) - GF_2(shares).sum())
    return shares

@dataclass
class SecBitN:
    # shares is a dictionary mapping each party to their share
    # each share is a located value in GF(2) (a bit)
    shares: Dict[pychor.Party, GF_2]

    @classmethod
    def input(cls, parties, val):
        """Secret share an input"""
        assert val.parties <= set(parties)
        assert len(val.parties) == 1
        owner = list(val.parties)[0]

        shares = share_bit(val, len(parties)).unlist(len(parties))
        shares_dict = {p: share for p, share in zip(parties, shares)}

        for p, share in shares_dict.items():
            share.send(owner, p)

        return SecBitN(shares_dict)


    def __add__(x, y):
        """Add two SecBitN objects using local addition of shares"""
        parties = x.shares.keys()
        assert y.shares.keys() == parties
        return SecBitN({p: x.shares[p] + y.shares[p] for p in parties})

    def __mul__(x, y):
        """Multiply two SecBitN objects using n-party GMW multiplication"""
        r = protocol_gmw_mult_n(x.shares, y.shares)
        return SecBitN(r)

    def reveal(self):
        """Reveal the secret value by broadcast and reconstruction"""
        @pychor.local_function
        def reconstruct(shares):
            return GF_2(shares).sum()

        parties = list(self.shares.keys())

        for p1 in parties:
            for p2 in parties:
                self.shares[p1].send(p1, p2)

        return reconstruct(list(self.shares.values()))

In [ ]:
with pychor.LocalBackend():
    parties = [pychor.Party(f'p{i}') for i in range(1, 5)]
    xv = parties[0].constant(GF_2(0))
    yv = parties[1].constant(GF_2(1))
    x = SecBitN.input(parties, xv)
    y = SecBitN.input(parties, yv)
    print('x object:', x)
    print('y object:', y)
    z = x + y
    print('z object:', z)
    print('z revealed:', z.reveal())

## Question 1 setup: OT

Below is the code for oblivious transfer from Chapter 6 of the textbook.

In [ ]:
from nacl.public import PrivateKey, PublicKey, SealedBox
from nacl.utils import random

def protocol_ot(sender, receiver, inputs, selection, n):
    # Function for the Receiver to generate keys
    @pychor.local_function
    def gen_keys(selection, n):
        # Generate a single real key pair key = (sk, pk)
        key = PrivateKey.generate()
        public_keys = [PublicKey(random(PublicKey.SIZE)) for _ in range(n)]
        public_keys[selection] = key.public_key
        return key, public_keys

    # Function for the Sender to encrypt the secret inputs
    @pychor.local_function
    def encrypt_inputs(pub_keys, inputs):
        # Encode the inputs as bytes
        length = max([(int(x).bit_length() + 7) // 8 for x in inputs])
        inputs_bytes = [int(x).to_bytes(length, 'little') for x in inputs]
    
        # Encrypt the inputs
        encrypted_inputs = [SealedBox(pk).encrypt(x) for pk, x in \
                            zip(pub_keys, inputs_bytes)]
        return encrypted_inputs

    # Function for the Receiver to decrypt the result
    @pychor.local_function
    def decrypt_result(selection, key, encrypted_inputs):
        # Select the correct input
        selected_input = encrypted_inputs[selection]
        # Decrypt it and convert it from bytes to int
        plaintext = SealedBox(key).decrypt(selected_input)
        return int.from_bytes(plaintext, 'little')

    # Step 1: Generate keys and send to Sender
    sk, pub_keys = gen_keys(selection, n).untup(2)
    pub_keys.send(receiver, sender)

    # Step 2: Encrypt inputs and send to Receiver
    encrypted_inputs = encrypt_inputs(pub_keys, inputs)
    encrypted_inputs.send(sender, receiver)

    # Step 3: Decrypt result
    result = decrypt_result(selection, sk, encrypted_inputs)

    return result

## Question 1 (30 points)

Implement the $n$-party GMW multiplication protocol. Reference [Chapter 7 of the textbook](https://jnear.github.io/programming-mpc/chapters/chapter07.html), the exercise from 2/9, and [slide 34 of the slides from UIUC CS 598](https://courses.grainger.illinois.edu/cs598man/sp2016/slides/17.pdf).

The inputs `x` and `y` will be dictionaries mapping parties to shares in GF(2) (bits). The output should be a dictionary mapping the same parties to shares of the product in GF(2).

Hint: the simplest approach is to directly implement the idea on slide 34, using one 1-out-of-2 OT per pair of parties. The code for OT 

In [ ]:
def protocol_gmw_mult_n(x_shares, y_shares):
    parties = list(x_shares.keys())
    assert list(y_shares.keys()) == parties

    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
with pychor.LocalBackend():
    parties = [pychor.Party(f'p{i}') for i in range(1, 5)]
    xv = parties[0].constant(GF_2(0))
    yv = parties[1].constant(GF_2(1))
    x = SecBitN.input(parties, xv)
    y = SecBitN.input(parties, yv)
    print('x*y revealed:', (x*y).reveal())
    print('y*y revealed:', (y*y).reveal())
    assert (x*y).reveal().val == 0
    assert (y*y).reveal().val == 1

## Question 2 (10 points)

Implement a protocol to generate an $n$-party binary multiplication triple. Reference [Chapter 6 of the textbook](https://jnear.github.io/programming-mpc/chapters/chapter06.html#application-generating-binary-multiplication-triples-using-ot).

Your solution should return three dictionaries, one for each of $a$, $b$, and $c$. Each dictionary should map parties to their shares of the value, and it should be the case that $ab = c$.

Hint: you can re-use `protocol_gmw_mult_n` above.

In [ ]:
def protocol_gen_binary_mult_triple_n(parties):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
def reconstruct_2(shares):
    parties = list(shares.keys())
    for p1 in parties:
        for p2 in parties:
            shares[p1].send(p1, p2)
    return pychor.locally(lambda x: GF_2(x).sum(), list(shares.values()))
    

with pychor.LocalBackend():
    parties = [pychor.Party(f'p{i}') for i in range(1, 5)]
    a, b, c = protocol_gen_binary_mult_triple_n(parties)
    av = reconstruct_2(a)
    bv = reconstruct_2(b)
    cv = reconstruct_2(c)
    print('a*b:', av*bv)
    print('c:', cv)
    assert (av*bv).val == cv.val

## Question 3 (30 points)

Implement a protocol to generate an $n$-party arithmetic multiplication triple. Reference [Chapter 6 of the textbook](https://jnear.github.io/programming-mpc/chapters/chapter06.html#application-generating-arithmetic-multiplication-triples-using-ot).

Your solution should return three dictionaries, one for each of $a$, $b$, and $c$. Each dictionary should map parties to their shares of the value, and it should be the case that $ab = c$.

Hint: you will probably want to use `protocol_crossterm` from the 2/9 exercise or Chapter 6; you'll have to make the participating parties `p1` and `p2` be arguments to the protocol so that you can run it pairwise between different parties. You might want to work out the 3-party example on paper first, to see what cross-terms appear. The structure will look similar to $n$-party GMW.

In [ ]:
def protocol_gen_arithmetic_mult_triple_n(parties):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
def reconstruct(shares):
    parties = list(shares.keys())
    for p1 in parties:
        for p2 in parties:
            shares[p1].send(p1, p2)
    return pychor.locally(lambda x: GF(x).sum(), list(shares.values()))
    

with pychor.LocalBackend():
    parties = [pychor.Party(f'p{i}') for i in range(1, 5)]
    a, b, c = protocol_gen_arithmetic_mult_triple_n(parties)
    av = reconstruct(a)
    bv = reconstruct(b)
    cv = reconstruct(c)
    print('a*b:', av*bv)
    print('c:', cv)
    assert (av*bv).val == cv.val